# 🧪 Module C: Soil Pattern Analysis using Unsupervised K-Means Clustering
*Smart Crop & Agriculture Analyzer*


In [ ]:
# =============================================================================
# 1. ENVIRONMENT SETUP & CORE LIBRARY IMPORTS
# =============================================================================
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn Preprocessing & Pipelines
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Scikit-Learn Models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, mean_absolute_error, mean_squared_error,
    r2_score, silhouette_score
)

# Plotting & Display configurations
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print("✅ All core Machine Learning and Data Science libraries imported successfully!")


## 1. Extract Soil Features & Determine Optimal k


In [ ]:
# 2.2 Inspect Crop Recommendation Data Quality
print("=== Crop Recommendation Dataset (First 5 Rows) ===")
display(crop_df.head())

print("\nMissing Values Check:")
print(crop_df.isnull().sum())

print(f"\nDuplicate Records: {crop_df.duplicated().sum()}")
print(f"Total Unique Crop Categories: {crop_df['label'].nunique()} crops")
print(f"Crop Categories: {sorted(crop_df['label'].unique().tolist())}")

print("\nClass Distribution (Balanced: 100 samples per crop):")
print(crop_df['label'].value_counts().head(5))


In [ ]:
# 6.1 Soil Feature Scaling & Optimal k Silhouette Analysis
soil_features = ["N", "P", "K", "ph"]
X_soil = crop_df[soil_features].copy()

soil_scaler = StandardScaler()
X_soil_scaled = soil_scaler.fit_transform(X_soil)

sil_scores = {}
for k in [3, 4, 5]:
    km_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km_test.fit_predict(X_soil_scaled)
    sil_scores[k] = silhouette_score(X_soil_scaled, labels)
    print(f"k = {k}: Silhouette Score = {sil_scores[k]:.4f}")

# Fit optimal K-Means model (k = 4)
optimal_k = 4
kmeans_soil = KMeans(n_clusters=optimal_k, init="k-means++", n_init=10, random_state=42)
cluster_labels = kmeans_soil.fit_predict(X_soil_scaled)
final_silhouette = silhouette_score(X_soil_scaled, cluster_labels)
print(f"\n✅ Selected k = {optimal_k} with Silhouette Score: {final_silhouette:.4f}")


## 2. Cluster Profiles & Agronomic Characteristics


In [ ]:
# 6.2 Profiling the 4 Discovered Soil Clusters
X_soil_profile = X_soil.copy()
X_soil_profile["Cluster"] = cluster_labels

cluster_summary = X_soil_profile.groupby("Cluster").agg(
    Record_Count=("N", "count"),
    Mean_N=("N", "mean"),
    Mean_P=("P", "mean"),
    Mean_K=("K", "mean"),
    Mean_pH=("ph", "mean")
).round(2).reset_index()

cluster_descriptions = {
    0: "Cluster 0: Legume & Neutral Soil (Moderate N, P, K, pH ~ 7.2) -> Ideal for pulses/chickpea",
    1: "Cluster 1: High Nitrogen Soil (High N ~ 96, pH ~ 6.5) -> Heavy feeders like rice, maize, banana",
    2: "Cluster 2: Acidic Plantation Soil (Low N ~ 22, pH ~ 5.8) -> Acid-tolerant trees like coconut, mango",
    3: "Cluster 3: Orchard Soil (Extremely High P ~ 133 & K ~ 200) -> Potassium-heavy fruits like grapes, apple"
}

cluster_summary["Agronomic_Interpretation"] = cluster_summary["Cluster"].map(cluster_descriptions)
print("=== Soil Cluster Characteristics (Original Measurement Units) ===")
display(cluster_summary[["Cluster", "Record_Count", "Mean_N", "Mean_P", "Mean_K", "Mean_pH", "Agronomic_Interpretation"]])


## 3. 2D PCA Visualization


In [ ]:
# 6.3 2D PCA Visualization of Soil Chemistry Clusters
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_soil_scaled)

pca_df = pd.DataFrame({
    "PCA1": X_pca[:, 0],
    "PCA2": X_pca[:, 1],
    "Cluster": cluster_labels
})

plt.figure(figsize=(8, 4.5))
colors = ["#2E7D32", "#1565C0", "#E65100", "#6A1B9A"]
for i in range(4):
    sub = pca_df[pca_df["Cluster"] == i]
    plt.scatter(sub["PCA1"], sub["PCA2"], c=colors[i], label=f"Cluster {i} (n={len(sub)})", alpha=0.45, s=35)

plt.title("2D PCA Visualization of Soil Chemistry Clusters (Standardized N, P, K, pH)", fontsize=11, fontweight="bold")
plt.xlabel(f"PCA 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PCA 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()


## 4. Soil Sample Cluster Predictor


In [ ]:
# 6.4 Function to Predict Cluster for New Soil Sample & Project to PCA
def analyze_soil_sample(n, p, k, ph):
    """Assigns new soil sample to a cluster and explains its agronomic profile."""
    sample_df = pd.DataFrame([[n, p, k, ph]], columns=["N", "P", "K", "ph"])
    sample_scaled = soil_scaler.transform(sample_df)
    
    assigned_cluster = int(kmeans_soil.predict(sample_scaled)[0])
    sample_pca = pca.transform(sample_scaled)[0]
    
    print(f"🧪 SOIL PATTERN ANALYSIS:")
    print(f"   • Assigned Cluster: {assigned_cluster}")
    print(f"   • Profile: {cluster_descriptions[assigned_cluster]}")
    print(f"   • PCA 2D Coordinates: ({sample_pca[0]:.3f}, {sample_pca[1]:.3f})")
    return assigned_cluster, sample_pca

print("--- TEST CASE 1: High Nitrogen Soil Sample ---")
analyze_soil_sample(n=100, p=45, k=40, ph=6.5)

print("\n--- TEST CASE 2: High Phosphorus & Potassium Orchard Soil ---")
analyze_soil_sample(n=25, p=135, k=200, ph=6.0)
